In [ ]:
pip install opencv-python ultralytics numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.3 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:


# -----------------------------
# Configuration
# -----------------------------
VIDEO_PATH = "traffic.mp4"  # replace with your video
OUTPUT_PATH = "output.mp4"
CONF_THRESHOLD = 0.4
VEHICLE_CLASSES = [2, 3, 5, 7]  # car, motorbike, bus, truck
DENSITY_THRESHOLD = 10

# Signal timing (seconds)
FIXED_GREEN = 20
MIN_GREEN = 10
MAX_GREEN = 60

# -----------------------------
# Load YOLO model
# -----------------------------
model = YOLO("yolov8n.pt")

# -----------------------------
# Helper functions
# -----------------------------
def estimate_density(count):
    if count < 5:
        return "Low"
    elif count < 10:
        return "Medium"
    else:
        return "High"


def adaptive_signal_time(count):
    green_time = MIN_GREEN + (count * 2)
    return min(MAX_GREEN, green_time)

# -----------------------------
# Video processing
# -----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)

# Get frame width & height
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 20

# Video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (frame_width, frame_height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO detection
    results = model(frame)[0]

    vehicle_count = 0

    for box in results.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if cls in VEHICLE_CLASSES and conf > CONF_THRESHOLD:
            vehicle_count += 1

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Density estimation
    density = estimate_density(vehicle_count)

    # Signal timings
    fixed_time = FIXED_GREEN
    adaptive_time = adaptive_signal_time(vehicle_count)

    # Display info
    cv2.putText(frame, f"Vehicles: {vehicle_count}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.putText(frame, f"Density: {density}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    cv2.putText(frame, f"Fixed Green: {fixed_time}s", (20, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

    cv2.putText(frame, f"Adaptive Green: {adaptive_time}s", (20, 160),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    # Congestion warning
    if vehicle_count > DENSITY_THRESHOLD:
        cv2.putText(frame, "CONGESTION ALERT!", (20, 220),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

    # Write frame to output video
    out.write(frame)

# Release resources
cap.release()
out.release()

print("\nProcessing complete. Output saved as output.mp4")

# -----------------------------
# Conceptual Comparison
# -----------------------------
print("\n--- Comparison ---")
print("Fixed-time signal: Same green time regardless of traffic")
print("Adaptive signal: Adjusts green time based on vehicle density")
print("Result: Adaptive reduces waiting time and congestion")




0: 384x640 8 persons, 2 bicycles, 5 cars, 5 motorcycles, 1 bus, 4 trucks, 10.2ms
Speed: 2.9ms preprocess, 10.2ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 7 cars, 1 motorcycle, 2 buss, 3 trucks, 8.6ms
Speed: 3.8ms preprocess, 8.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 10 cars, 4 motorcycles, 1 bus, 3 trucks, 8.6ms
Speed: 3.9ms preprocess, 8.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 persons, 1 bicycle, 7 cars, 4 motorcycles, 4 buss, 1 truck, 9.3ms
Speed: 3.0ms preprocess, 9.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 15 persons, 2 bicycles, 7 cars, 3 motorcycles, 4 trucks, 9.9ms
Speed: 3.8ms preprocess, 9.9ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 persons, 2 bicycles, 9 cars, 4 motorcycles, 2 buss, 1 truck, 1 traffic light, 9.9ms
Speed: 3.9ms preprocess, 9.9ms inference, 1